In [5]:
%pip install chardet

Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd

# Option 1: Windows-1252 (common for CSVs on Windows)
spam_df = pd.read_csv("C:\\Users\\user\\OneDrive\\Desktop\\Project - NLP\\spam_email_dataset.csv", encoding="cp1252")

# Option 2: ISO-8859-1
# spam_df = pd.read_csv("C:\\Users\\user\\OneDrive\\Desktop\\Project - NLP\\spam_email_dataset.csv", encoding="ISO-8859-1")

# Option 3: Detect encoding automatically
import chardet
with open("C:\\Users\\user\\OneDrive\\Desktop\\Project - NLP\\spam_email_dataset.csv", "rb") as f:
    result = chardet.detect(f.read(100000))
print(result)

spam_df = pd.read_csv("C:\\Users\\user\\OneDrive\\Desktop\\Project - NLP\\spam_email_dataset.csv", encoding=result['encoding'])


{'encoding': 'Windows-1252', 'confidence': 0.2507450261995026, 'language': 'en', 'mime_type': 'text/plain'}


In [7]:
emotion_df = pd.read_csv(
    "C:\\Users\\user\\OneDrive\\Desktop\\Project - NLP\\final_dataset (1).csv",
    encoding="ISO-8859-1"
)



In [8]:
print("Spam dataset columns:", spam_df.columns)
print("Emotion dataset columns:", emotion_df.columns)


Spam dataset columns: Index(['Subject', 'Body', 'Spam Label'], dtype='object')
Emotion dataset columns: Index(['text', 'emotion'], dtype='object')


In [9]:
# Rename columns for consistency
spam_df.rename(columns={
    'Subject':'subject',
    'Body':'text',          # main email body becomes 'text'
    'Spam Label':'label'    # spam/ham indicator becomes 'label'
}, inplace=True)

emotion_df.rename(columns={
    'text':'text',          # already correct
    'emotion':'label'       # rename to 'label' for consistency
}, inplace=True)

import nltk
import re
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Helper: map POS tags to WordNet tags
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

# Main preprocessing function
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)   # remove punctuation/numbers
    tokens = nltk.word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]

    pos_tags = nltk.pos_tag(tokens)
    lemmatized = [lemmatizer.lemmatize(w, get_wordnet_pos(tag)) for w, tag in pos_tags]

    return " ".join(lemmatized)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [10]:
# Standardize column names
spam_df.rename(columns={'Subject':'subject','Body':'text','Spam Label':'label'}, inplace=True)
emotion_df.rename(columns={'text':'text','emotion':'label'}, inplace=True)

# Create clean_text column
spam_df['clean_text'] = spam_df['text'].apply(preprocess_text)
emotion_df['clean_text'] = emotion_df['text'].apply(preprocess_text)


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X_spam = spam_df['clean_text']
y_spam = spam_df['label']

X_train, X_test, y_train, y_test = train_test_split(X_spam, y_spam, test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

spam_model = LogisticRegression(max_iter=1000)
spam_model.fit(X_train_vec, y_train)

y_pred_spam = spam_model.predict(X_test_vec)
print("Spam Detection Results:\n", classification_report(y_test, y_pred_spam))


Spam Detection Results:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        97
           1       1.00      1.00      1.00       203

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300



In [12]:
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense

# Encode labels
encoder = LabelEncoder()
y_emotion = encoder.fit_transform(emotion_df['label'])

# Tokenize text
tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(emotion_df['clean_text'])
X_seq = tokenizer.texts_to_sequences(emotion_df['clean_text'])
X_pad = pad_sequences(X_seq, maxlen=100)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_pad, y_emotion, test_size=0.2, random_state=42)

# Build BiLSTM model
model = Sequential([
    Embedding(input_dim=10000, output_dim=128, input_length=100),
    Bidirectional(LSTM(64)),
    Dense(64, activation='relu'),
    Dense(len(encoder.classes_), activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=5, batch_size=32)

# Evaluate
loss, acc = model.evaluate(X_test, y_test)
print(f"Emotion Classification Accuracy: {acc:.2f}")


Epoch 1/5


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


2659/2659 ━━━━━━━━━━━━━━━━━━━━ 78s 29ms/step - accuracy: 0.7839 - loss: 0.7007 - val_accuracy: 0.8532 - val_loss: 0.4643
Epoch 2/5
2659/2659 ━━━━━━━━━━━━━━━━━━━━ 79s 30ms/step - accuracy: 0.8751 - loss: 0.3929 - val_accuracy: 0.8615 - val_loss: 0.4413
Epoch 3/5
2659/2659 ━━━━━━━━━━━━━━━━━━━━ 82s 31ms/step - accuracy: 0.8941 - loss: 0.3273 - val_accuracy: 0.8622 - val_loss: 0.4547
Epoch 4/5
2659/2659 ━━━━━━━━━━━━━━━━━━━━ 80s 30ms/step - accuracy: 0.9101 - loss: 0.2743 - val_accuracy: 0.8615 - val_loss: 0.4961
Epoch 5/5
2659/2659 ━━━━━━━━━━━━━━━━━━━━ 74s 28ms/step - accuracy: 0.9244 - loss: 0.2252 - val_accuracy: 0.8561 - val_loss: 0.5441
665/665 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.8561 - loss: 0.5441
Emotion Classification Accuracy: 0.86


In [19]:
import joblib

# Save spam detection model and vectorizer
joblib.dump(spam_model, "spam_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

# Save emotion detection model and tokenizer
model.save("emotion_model.h5")
joblib.dump(tokenizer, "tokenizer.pkl")
joblib.dump(encoder, "label_encoder.pkl")



['label_encoder.pkl']

In [1]:
try:
    import joblib
    print("Joblib version:", joblib.__version__)
except ImportError:
    print("Joblib not installed in this environment")


Joblib version: 1.5.3
